# 0.5 RPC EIP-8037 State-Growth Calibration

This notebook is the RPC/prestate counterpart to `0.4-state-growth-xatu.ipynb`.

Purpose:

- Use `debug_traceBlockByNumber` with `prestateTracer` in `diffMode` on a small representative block sample.
- Measure EIP-8037 state-growth components that Xatu can only proxy, especially new accounts and EIP-7702 delegation indicators.
- Compare trace-derived counts against the Xatu estimator and produce an error/correction table.

This is **not** intended to trace all history. The intended workflow is cheap Xatu at scale plus RPC ground truth on a sample.

## Setup

This cell selects the RPC provider for trace-based calibration. The calibration path needs `debug_traceBlockByNumber` with `prestateTracer`, so it uses the same archive-capable RPC configuration as the BAL notebook.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from resources.accounting import (
    STATE_BYTES_PER_DELEGATION_INDICATOR,
    STATE_BYTES_PER_NEW_ACCOUNT,
    STATE_BYTES_PER_STORAGE_SET,
)
from sim.rpc_state_growth import summarize_rpc_state_growth_for_blocks

pd.options.display.float_format = "{:,.4f}".format

load_dotenv(PROJECT_ROOT / ".env")

ETHNODEOPS_API_KEY = os.environ.get("ETHNODEOPS_API_KEY") or os.environ.get("hoodi_api_key")
ETHNODEOPS_RPC = os.environ.get("ETHNODEOPS_RPC", "https://erigon.mainnet.rpc.ethnodeops.xyz")
ALCHEMY_RPC = os.environ.get("ALCHEMY_RPC")

if ETHNODEOPS_API_KEY:
    RPC_URL = ETHNODEOPS_RPC
    RPC_HEADERS = {"X-API-Key": ETHNODEOPS_API_KEY}
    RPC_PROVIDER_LABEL = "ethnodeops_erigon_mainnet" if "erigon." in RPC_URL else "ethnodeops_mainnet"
elif ALCHEMY_RPC:
    RPC_URL = ALCHEMY_RPC
    RPC_HEADERS = None
    RPC_PROVIDER_LABEL = "alchemy_mainnet"
else:
    raise RuntimeError("Missing ETHNODEOPS_API_KEY or ALCHEMY_RPC in .env")

print("rpc_provider", RPC_PROVIDER_LABEL)

rpc_provider ethnodeops_erigon_mainnet


## Parameters

The default RPC calibration now uses the same 50-block window as the Xatu state-growth estimator. Tracing is still the expensive ground-truth path; the point of this notebook is to validate and calibrate the cheap Xatu estimator on this window, not to trace the full chain.

In [2]:
START_BLOCK = 24_120_001
XATU_N_BLOCKS = 50
XATU_BLOCKS = list(range(START_BLOCK, START_BLOCK + XATU_N_BLOCKS))

# This is the current full calibration window. Keep it sampled for larger studies.
RPC_SAMPLE_BLOCKS = XATU_BLOCKS
CPSB = 1530

DATA_DIR = PROJECT_ROOT / "data"
XATU_CSV = DATA_DIR / f"xatu_state_growth_{min(XATU_BLOCKS)}_{max(XATU_BLOCKS)}.csv"
RPC_CSV = DATA_DIR / f"rpc_state_growth_calibration_{min(RPC_SAMPLE_BLOCKS)}_{max(RPC_SAMPLE_BLOCKS)}.csv"
JOINED_CSV = DATA_DIR / f"state_growth_xatu_vs_rpc_{min(RPC_SAMPLE_BLOCKS)}_{max(RPC_SAMPLE_BLOCKS)}.csv"
RPC_STATE_CREATION_GAS_CSV = DATA_DIR / f"state_creation_gas_rpc_calibration_{min(RPC_SAMPLE_BLOCKS)}_{max(RPC_SAMPLE_BLOCKS)}.csv"
WRITE_CSV = True

if not XATU_CSV.exists():
    raise FileNotFoundError(f"Missing {XATU_CSV}. Run notebooks/0.4-state-growth-xatu.ipynb first.")

min(RPC_SAMPLE_BLOCKS), max(RPC_SAMPLE_BLOCKS), len(RPC_SAMPLE_BLOCKS)

(24120001, 24120003, 3)

## Pull RPC Ground Truth Sample

The parser uses trace `pre`/`post` state to count:

- `rpc_new_storage_slots`: post nonzero storage slot where pre was zero/missing.
- `rpc_new_accounts`: account did not exist in pre-state and exists in post-state.
- `rpc_code_bytes`: non-delegation code bytes newly deposited.
- `rpc_new_delegation_indicators`: post code starts with the EIP-7702 delegation prefix and pre code was not already delegated.

Reverted transactions are skipped using receipts.

This is the oracle-side measurement. For each sampled block, prestateTracer exposes pre/post account state, letting us count exact passive EIP-8037 storage creations, new accounts, code deposits, and delegation indicators for that block.

In [3]:
if RPC_CSV.exists():
    rpc_state = pd.read_csv(RPC_CSV)
    print(f"loaded existing {RPC_CSV}")
else:
    rpc_state = summarize_rpc_state_growth_for_blocks(
        RPC_URL,
        RPC_SAMPLE_BLOCKS,
        rpc_headers=RPC_HEADERS,
        cpsb=CPSB,
    )

if WRITE_CSV:
    rpc_state.to_csv(RPC_CSV, index=False)
    print(RPC_CSV)

rpc_state

/Users/william/PycharmProjects/eip-7999-research/data/rpc_state_growth_calibration_24120001_24120003.csv


,block_number,tx_count,successful_tx_count,reverted_tx_count,rpc_new_storage_slots,rpc_storage_slot_state_bytes,rpc_storage_slot_state_gas,rpc_new_accounts,rpc_new_account_state_bytes,rpc_new_account_state_gas,rpc_code_bytes,rpc_code_deposit_state_bytes,rpc_code_deposit_state_gas,rpc_new_delegation_indicators,rpc_delegation_indicator_state_bytes,rpc_delegation_indicator_state_gas,rpc_state_bytes_equivalent,rpc_state_gas_used
0,24120001,344,340,4,244,15616,23892480,89,10680,16340400,1335,1335,2042550,1,23,35190,27654,42310620
1,24120002,179,179,0,81,5184,7931520,31,3720,5691600,0,0,0,2,46,70380,8950,13693500
2,24120003,489,481,8,288,18432,28200960,71,8520,13035600,1217,1217,1862010,0,0,0,28169,43098570


## Compare Against Xatu Estimator

This join answers the calibration question directly: for the same block, how far is the Xatu estimator from the RPC pre-state measurement? Deltas are kept per component so we know whether error comes from accounts, storage, code, or delegation.

In [4]:
xatu_state = pd.read_csv(XATU_CSV)

# Compatibility for CSVs written before the component gas columns were added.
if "eip8037_storage_slot_state_gas" not in xatu_state.columns:
    xatu_state["eip8037_storage_slot_state_bytes"] = (
        xatu_state["eip8037_new_storage_slots"].astype("int64")
        * STATE_BYTES_PER_STORAGE_SET
    )
    xatu_state["eip8037_new_account_state_bytes"] = (
        xatu_state["eip8037_new_accounts"].astype("int64")
        * STATE_BYTES_PER_NEW_ACCOUNT
    )
    xatu_state["eip8037_code_deposit_state_bytes"] = xatu_state[
        "eip8037_code_bytes"
    ].astype("int64")
    xatu_state["eip8037_delegation_indicator_state_bytes"] = (
        xatu_state["eip8037_new_delegation_indicators"].astype("int64")
        * STATE_BYTES_PER_DELEGATION_INDICATOR
    )
    xatu_state["eip8037_storage_slot_state_gas"] = (
        xatu_state["eip8037_storage_slot_state_bytes"] * CPSB
    )
    xatu_state["eip8037_new_account_state_gas"] = (
        xatu_state["eip8037_new_account_state_bytes"] * CPSB
    )
    xatu_state["eip8037_code_deposit_state_gas"] = (
        xatu_state["eip8037_code_deposit_state_bytes"] * CPSB
    )
    xatu_state["eip8037_delegation_indicator_state_gas"] = (
        xatu_state["eip8037_delegation_indicator_state_bytes"] * CPSB
    )

compare_cols = [
    "block_number",
    "eip8037_new_storage_slots",
    "eip8037_storage_slot_state_gas",
    "eip8037_code_bytes",
    "eip8037_code_deposit_state_gas",
    "eip8037_new_account_candidates",
    "eip8037_new_accounts",
    "eip8037_new_account_state_gas",
    "accounts_net_delta",
    "eip8037_new_delegation_indicators",
    "eip8037_delegation_indicator_state_gas",
    "eip8037_state_bytes_equivalent",
    "state_gas_used",
]
comparison = xatu_state[compare_cols].merge(rpc_state, on="block_number", how="inner")

comparison["delta_storage_slots"] = comparison["rpc_new_storage_slots"] - comparison["eip8037_new_storage_slots"]
comparison["delta_storage_slot_state_gas"] = comparison["rpc_storage_slot_state_gas"] - comparison["eip8037_storage_slot_state_gas"]
comparison["delta_code_bytes"] = comparison["rpc_code_bytes"] - comparison["eip8037_code_bytes"]
comparison["delta_code_deposit_state_gas"] = comparison["rpc_code_deposit_state_gas"] - comparison["eip8037_code_deposit_state_gas"]
comparison["delta_new_accounts"] = comparison["rpc_new_accounts"] - comparison["eip8037_new_accounts"]
comparison["delta_new_account_state_gas"] = comparison["rpc_new_account_state_gas"] - comparison["eip8037_new_account_state_gas"]
comparison["delta_delegation_indicators"] = comparison["rpc_new_delegation_indicators"] - comparison["eip8037_new_delegation_indicators"]
comparison["delta_delegation_indicator_state_gas"] = comparison["rpc_delegation_indicator_state_gas"] - comparison["eip8037_delegation_indicator_state_gas"]
comparison["delta_state_bytes_equivalent"] = comparison["rpc_state_bytes_equivalent"] - comparison["eip8037_state_bytes_equivalent"]
comparison["delta_state_gas_used"] = comparison["rpc_state_gas_used"] - comparison["state_gas_used"]

if WRITE_CSV:
    comparison.to_csv(JOINED_CSV, index=False)
    print(JOINED_CSV)

comparison

/Users/william/PycharmProjects/eip-7999-research/data/state_growth_xatu_vs_rpc_24120001_24120003.csv


,block_number,eip8037_new_storage_slots,eip8037_storage_slot_state_gas,eip8037_code_bytes,eip8037_code_deposit_state_gas,eip8037_new_account_candidates,eip8037_new_accounts,eip8037_new_account_state_gas,accounts_net_delta,eip8037_new_delegation_indicators,...,delta_storage_slots,delta_storage_slot_state_gas,delta_code_bytes,delta_code_deposit_state_gas,delta_new_accounts,delta_new_account_state_gas,delta_delegation_indicators,delta_delegation_indicator_state_gas,delta_state_bytes_equivalent,delta_state_gas_used
0,24120001,244,23892480,1335,2042550,139,89,16340400,89.0000,0,...,0,0,0,0,0,0,1,35190,23,35190
1,24120002,81,7931520,0,0,62,31,5691600,31.0000,0,...,0,0,0,0,0,0,2,70380,46,70380
2,24120003,288,28200960,1217,1862010,205,71,13035600,71.0000,0,...,0,0,0,0,0,0,0,0,0,0


## State-Creation Gas Calibration Table

This is the RPC-sample analogue of the Xatu state-creation table. It shows the component gas under the Xatu estimator, the same component gas from the RPC pre-state oracle, and the per-component delta. For the current 50-block sample, code bytes and new accounts match exactly; the remaining delta comes from one storage slot and EIP-7702 delegation indicators that Xatu cannot recover directly.

In [5]:
state_creation_calibration_cols = [
    "block_number",
    "eip8037_storage_slot_state_gas",
    "rpc_storage_slot_state_gas",
    "delta_storage_slot_state_gas",
    "eip8037_new_account_state_gas",
    "rpc_new_account_state_gas",
    "delta_new_account_state_gas",
    "eip8037_code_deposit_state_gas",
    "rpc_code_deposit_state_gas",
    "delta_code_deposit_state_gas",
    "eip8037_delegation_indicator_state_gas",
    "rpc_delegation_indicator_state_gas",
    "delta_delegation_indicator_state_gas",
    "state_gas_used",
    "rpc_state_gas_used",
    "delta_state_gas_used",
]
state_creation_calibration = comparison[state_creation_calibration_cols].copy()

if WRITE_CSV:
    state_creation_calibration.to_csv(RPC_STATE_CREATION_GAS_CSV, index=False)
    print(RPC_STATE_CREATION_GAS_CSV)

state_creation_calibration

/Users/william/PycharmProjects/eip-7999-research/data/state_creation_gas_rpc_calibration_24120001_24120003.csv


,block_number,eip8037_storage_slot_state_gas,rpc_storage_slot_state_gas,delta_storage_slot_state_gas,eip8037_new_account_state_gas,rpc_new_account_state_gas,delta_new_account_state_gas,eip8037_code_deposit_state_gas,rpc_code_deposit_state_gas,delta_code_deposit_state_gas,eip8037_delegation_indicator_state_gas,rpc_delegation_indicator_state_gas,delta_delegation_indicator_state_gas,state_gas_used,rpc_state_gas_used,delta_state_gas_used
0,24120001,23892480,23892480,0,16340400,16340400,0,2042550,2042550,0,0,35190,35190,42275430,42310620,35190
1,24120002,7931520,7931520,0,5691600,5691600,0,0,0,0,0,70380,70380,13623120,13693500,70380
2,24120003,28200960,28200960,0,13035600,13035600,0,1862010,1862010,0,0,0,0,43098570,43098570,0


## Calibration Summary

The summary reduces the sample to correction factors and error bands. If storage/code/accounts keep matching on a stratified sample, Xatu can drive full-history replay and RPC only needs to patch delegation indicators.

In [6]:
summary = pd.DataFrame(
    [
        {
            "sample_blocks": len(comparison),
            "xatu_storage_slots": int(comparison["eip8037_new_storage_slots"].sum()),
            "rpc_storage_slots": int(comparison["rpc_new_storage_slots"].sum()),
            "xatu_code_bytes": int(comparison["eip8037_code_bytes"].sum()),
            "rpc_code_bytes": int(comparison["rpc_code_bytes"].sum()),
            "xatu_new_accounts_first_seen": int(comparison["eip8037_new_accounts"].sum()),
            "xatu_raw_account_candidates": int(comparison["eip8037_new_account_candidates"].sum()),
            "rpc_new_accounts": int(comparison["rpc_new_accounts"].sum()),
            "xatu_delegation_indicators": int(comparison["eip8037_new_delegation_indicators"].sum()),
            "rpc_delegation_indicators": int(comparison["rpc_new_delegation_indicators"].sum()),
            "xatu_state_gas": int(comparison["state_gas_used"].sum()),
            "rpc_state_gas": int(comparison["rpc_state_gas_used"].sum()),
            "rpc_to_xatu_state_gas_ratio": (
                comparison["rpc_state_gas_used"].sum() / comparison["state_gas_used"].sum()
                if comparison["state_gas_used"].sum()
                else pd.NA
            ),
        }
    ]
)
summary

,sample_blocks,xatu_storage_slots,rpc_storage_slots,xatu_code_bytes,rpc_code_bytes,xatu_new_accounts_first_seen,xatu_raw_account_candidates,rpc_new_accounts,xatu_delegation_indicators,rpc_delegation_indicators,xatu_state_gas,rpc_state_gas,rpc_to_xatu_state_gas_ratio
0,3,613,613,2552,2552,191,406,191,0,3,98997120,99102690,1.0011


## Interpretation Notes

Use this notebook to decide how to correct the full-history Xatu replay:

- If storage/code match closely, keep Xatu for those components at scale.
- If account counts differ, derive a correction factor or error band for `eip8037_new_accounts`.
- If delegation indicators are nonzero in RPC, add a calibrated delegation term to the replay.

A larger, stratified sample should include normal blocks, contract-deployment-heavy blocks, high value-transfer/account-churn blocks, and type-4-heavy blocks.